# 🦀 Day 7: Mini-Project — CSV Parser

Parse CSV data with proper error handling for every failure mode!

---

## 🏗️ Step 1: Error Types

In [ ]:
use std::fmt;
use std::collections::HashMap;

#[derive(Debug)]
enum CsvError {
    EmptyInput,
    NoHeaders,
    ColumnMismatch { row: usize, expected: usize, got: usize },
    ParseError { row: usize, column: String, value: String, kind: String },
    MissingColumn(String),
}

impl fmt::Display for CsvError {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        match self {
            CsvError::EmptyInput => write!(f, "CSV input is empty"),
            CsvError::NoHeaders => write!(f, "No header row found"),
            CsvError::ColumnMismatch { row, expected, got } =>
                write!(f, "Row {}: expected {} columns, got {}", row, expected, got),
            CsvError::ParseError { row, column, value, kind } =>
                write!(f, "Row {}, column '{}': cannot parse '{}' as {}", row, column, value, kind),
            CsvError::MissingColumn(name) =>
                write!(f, "Column '{}' not found in headers", name),
        }
    }
}

impl std::error::Error for CsvError {}

println!("CsvError defined! ✅");

## 📊 Step 2: CSV Parser

In [ ]:
#[derive(Debug)]
struct CsvTable {
    headers: Vec<String>,
    rows: Vec<Vec<String>>,
}

impl CsvTable {
    fn parse(input: &str) -> Result<Self, CsvError> {
        let input = input.trim();
        if input.is_empty() {
            return Err(CsvError::EmptyInput);
        }

        let mut lines = input.lines();

        let header_line = lines.next().ok_or(CsvError::NoHeaders)?;
        let headers: Vec<String> = header_line.split(',').map(|s| s.trim().to_string()).collect();

        if headers.is_empty() {
            return Err(CsvError::NoHeaders);
        }

        let mut rows = Vec::new();
        for (i, line) in lines.enumerate() {
            let row: Vec<String> = line.split(',').map(|s| s.trim().to_string()).collect();
            if row.len() != headers.len() {
                return Err(CsvError::ColumnMismatch {
                    row: i + 2,  // 1-indexed, +1 for header
                    expected: headers.len(),
                    got: row.len(),
                });
            }
            rows.push(row);
        }

        Ok(CsvTable { headers, rows })
    }

    fn column_index(&self, name: &str) -> Result<usize, CsvError> {
        self.headers.iter().position(|h| h == name)
            .ok_or_else(|| CsvError::MissingColumn(name.to_string()))
    }

    fn get(&self, row: usize, column: &str) -> Result<&str, CsvError> {
        let col_idx = self.column_index(column)?;
        Ok(&self.rows[row][col_idx])
    }

    fn column_as_f64(&self, name: &str) -> Result<Vec<f64>, CsvError> {
        let idx = self.column_index(name)?;
        self.rows.iter().enumerate().map(|(i, row)| {
            row[idx].parse::<f64>().map_err(|_| CsvError::ParseError {
                row: i + 2,
                column: name.to_string(),
                value: row[idx].clone(),
                kind: "number".to_string(),
            })
        }).collect()
    }

    fn row_count(&self) -> usize { self.rows.len() }
    fn col_count(&self) -> usize { self.headers.len() }
}

println!("CsvTable parser built! ✅");

## 🧪 Step 3: Test It

In [ ]:
let csv_data = "name, age, score, grade
Alice, 25, 92.5, A
Bob, 30, 88.0, B+
Charlie, 22, 95.3, A+
Diana, 28, 79.8, B
Eve, 35, 91.1, A-";

match CsvTable::parse(csv_data) {
    Ok(table) => {
        println!("✅ Parsed {} rows × {} columns", table.row_count(), table.col_count());
        println!("Headers: {:?}", table.headers);

        for i in 0..table.row_count() {
            let name = table.get(i, "name").unwrap();
            let score = table.get(i, "score").unwrap();
            println!("  {} scored {}", name, score);
        }
    }
    Err(e) => println!("❌ {}", e),
}

In [ ]:
// Statistics on numeric columns
let table = CsvTable::parse(csv_data).unwrap();

match table.column_as_f64("score") {
    Ok(scores) => {
        let sum: f64 = scores.iter().sum();
        let avg = sum / scores.len() as f64;
        let max = scores.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
        let min = scores.iter().cloned().fold(f64::INFINITY, f64::min);

        println!("\n📊 Score Statistics:");
        println!("  Count: {}", scores.len());
        println!("  Sum:   {:.1}", sum);
        println!("  Avg:   {:.1}", avg);
        println!("  Max:   {:.1}", max);
        println!("  Min:   {:.1}", min);
    }
    Err(e) => println!("Error: {}", e),
}

## ❌ Step 4: Error Handling Tests

In [ ]:
// Test 1: Empty input
match CsvTable::parse("") {
    Err(CsvError::EmptyInput) => println!("✅ Empty input detected"),
    other => println!("❌ Expected EmptyInput, got: {:?}", other),
}

// Test 2: Column mismatch
let bad_csv = "a,b,c\n1,2\n4,5,6";
match CsvTable::parse(bad_csv) {
    Err(CsvError::ColumnMismatch { row, expected, got }) => 
        println!("✅ Column mismatch at row {}: expected {}, got {}", row, expected, got),
    other => println!("❌ Expected ColumnMismatch, got: {:?}", other),
}

// Test 3: Parse error
let num_csv = "value\n42\nabc\n99";
let table = CsvTable::parse(num_csv).unwrap();
match table.column_as_f64("value") {
    Err(CsvError::ParseError { row, column, value, .. }) =>
        println!("✅ Parse error at row {}, col '{}': '{}'", row, column, value),
    other => println!("❌ Expected ParseError, got: {:?}", other),
}

// Test 4: Missing column
match table.get(0, "nonexistent") {
    Err(CsvError::MissingColumn(name)) =>
        println!("✅ Missing column: '{}'", name),
    other => println!("❌ Expected MissingColumn, got: {:?}", other),
}

println!("\n🎉 All error cases handled!");

## 📋 Step 5: Pretty Printer

In [ ]:
impl fmt::Display for CsvTable {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        // Calculate column widths
        let widths: Vec<usize> = (0..self.headers.len()).map(|i| {
            let header_len = self.headers[i].len();
            let max_data = self.rows.iter().map(|r| r[i].len()).max().unwrap_or(0);
            header_len.max(max_data) + 2
        }).collect();

        // Header
        let separator: String = widths.iter().map(|w| "-".repeat(*w)).collect::<Vec<_>>().join("+");
        writeln!(f, "+{}+", separator)?;

        let header: String = self.headers.iter().enumerate()
            .map(|(i, h)| format!("{:^width$}", h, width = widths[i]))
            .collect::<Vec<_>>().join("|");
        writeln!(f, "|{}|", header)?;
        writeln!(f, "+{}+", separator)?;

        // Data rows
        for row in &self.rows {
            let line: String = row.iter().enumerate()
                .map(|(i, v)| format!("{:^width$}", v, width = widths[i]))
                .collect::<Vec<_>>().join("|");
            writeln!(f, "|{}|", line)?;
        }
        write!(f, "+{}+", separator)
    }
}

let table = CsvTable::parse(csv_data).unwrap();
println!("{}", table);

## 🏋️ Extension Challenges

In [ ]:
// Challenge 1: Add support for quoted fields ("hello, world")
// Challenge 2: Add filter_rows(column, value) method
// Challenge 3: Add sort_by(column) method for string and numeric sorting

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **Custom errors** with context (row, column, value) for precise diagnostics
2. **Match on specific errors** to test each failure mode
3. **`?` operator** chains operations cleanly inside methods
4. **Combinators** (`.map_err()`) transform errors at boundaries
5. **Display trait** for both human-readable output and error messages

---
**🎉 Month 2 Complete!** Next: Month 3 — Intermediate Rust!